### Day 14 · Pandas 进阶(分组聚合 / 合并 / 拼接 / 透视表 / 缺失值)

**Pandas 进阶**在 DataFrame 筛选的基础上,掌握 **groupby 分组聚合、merge 表合并、concat 拼接、pivot_table 透视表、缺失值处理**等核心技能,这是数据分析从"问一列"到"问整张大表"的关键一步。

前置:Day13 已掌握 DataFrame 创建、查看、loc/iloc 选取。

#### groupby + agg(列表 + 命名聚合)

痛点:查"每位学生平均分"容易,但如果想"按班级分组"再"同时看平均分、最高分、人数",一行语句怎么写?

类比:班级大合影时,老师让"按小组站好"(split),每组统计人数(apply),最后汇总一张清单(combine)。这就是 **拆分 → 应用 → 合并**三步曲。

解释: **groupby** 先把行按某列值分组;**agg** 再对每组一次性应用多个聚合函数。传入函数列表自动生成列名;使用 **命名聚合**自定义新列名,格式 `新列=("原列", "函数")`。

In [ ]:
# 导入 pandas,约定别名 pd
import pandas as pd

# 创建成绩表,包含班级、姓名、成绩三列
df = pd.DataFrame({
    "班级": ["A", "A", "B", "B", "B"],
    "姓名": ["张三", "李四", "王五", "赵六", "孙七"],
    "成绩": [88, 92, 75, 85, 90]
})

# --- 演练:按分组聚合 ---
# 先分组,再选列,最后调用 mean
result = df.groupby("班级")["成绩"].mean()
print(result)

# --- 执行过程 ---
# ① df.groupby("班级"):按"班级"列把行拆成 A 组、B 组
# ② ["成绩"]:从每组里只取"成绩"列,丢弃其他列
# ③ .mean():对每组的"成绩"求平均值,combine 成一行
# 最终每班只输出一行:索引是班级,值是平均成绩

**逐行解剖**

- `groupby("班级")`:按"班级"列的唯一值把 DataFrame 拆成若干子 DataFrame,返回 GroupBy 对象,此时还没有任何计算。
- `["成绩"]`:在 GroupBy 对象上选列,等价于告诉 Pandas "我只对成绩这一列做聚合"。如果不选列,会对所有数值列同时聚合。
- `.mean()`:对每组调用 mean()求均值,返回一个 Series,索引是分组键(班级),值是均值。

In [ ]:
# 导入 pandas
import pandas as pd

# 创建成绩表
df = pd.DataFrame({
    "班级": ["A", "A", "B", "B", "B"],
    "姓名": ["张三", "李四", "王五", "赵六", "孙七"],
    "成绩": [88, 92, 75, 85, 90]
})

# agg + 函数列表:一次性计算多个指标
result_list = df.groupby("班级")["成绩"].agg(["mean", "max", "min", "count"])
print(result_list)
print("---")

# 命名聚合:自定义输出列名,格式为 新列名=("原列名", "函数名")
result_named = df.groupby("班级").agg(
    平均成绩=("成绩", "mean"),
    最高分=("成绩", "max"),
    人数=("成绩", "count")
)
print(result_named)

# --- 执行过程 ---
# ① df.groupby("班级"):按班级分组
# ② ["成绩"].agg(["mean","max","min","count"]):对成绩列同时调用四种函数,生成多列
# ③ agg(新列=("原列","函数")):命名聚合,结果列名由我们自定,而不是函数名

**逐行解剖**

- `agg(["mean", "max", "min", "count"])`:**agg** 接收函数名列表,每个函数独立作用,结果列名就是函数名。适合快速看分布。
- `agg(平均成绩=("成绩", "mean"))`:**命名聚合**(Pandas 0.25+推荐),左值是新列名,右值是元组 ("被聚合列", "函数名")。多个命名聚合可以写在同一行,用逗号分隔。
- "count" vs "size":count 排除 NaN,size 计入 NaN,缺数据时要注意。

**练习**

超市销售表已有"品类"和"销售额"两列,请用 groupby + agg 同时输出每个品类的:总销售额(sum)、平均销售额(mean)、订单数(count)。

> 问自己:
> - 命名聚合的元组中,"原列"应该是哪一列?
> - 如果想看订单数量而不是金额,应该用 count 还是 size?

In [ ]:
# 学员代码区 - 请在此编写你的解答
import pandas as pd

df = pd.DataFrame({
    "品类": ["水果", "水果", "零食", "零食", "饮料"],
    "销售额": [120, 150, 80, 60, 95]
})

# 请用命名聚合输出:总销售额、平均销售额、订单数
pass

In [ ]:
# 参考答案
import pandas as pd

df = pd.DataFrame({
    "品类": ["水果", "水果", "零食", "零食", "饮料"],
    "销售额": [120, 150, 80, 60, 95]
})

# 按品类分组,对销售额做三种聚合
result = df.groupby("品类").agg(
    总销售额=("销售额", "sum"),
    平均销售额=("销售额", "mean"),
    订单数=("销售额", "count")
)
print(result)

#### merge(inner / left / right / outer)

痛点:学生信息在一个表、成绩在另一个表,如何把它们像 Excel VLOOKUP 一样拼起来?

类比:两张名单通过"学号"这条线左右对齐:inner 只留都有的,left 以左边为准右边没就空着,outer 全都要。

解释: **merge** 基于共同列(key)把两张表合并。**how** 参数控制连接方式:inner、left、right、outer。

In [ ]:
# 导入 pandas
import pandas as pd

# 左表:学生基本信息,学号 S1-S4
df_stu = pd.DataFrame({
    "学号": ["S1", "S2", "S3", "S4"],
    "姓名": ["张三", "李四", "王五", "赵六"]
})

# 右表:学生成绩,学号 S1、S2、S3、S5(S4 没成绩,S5 没基本信息)
df_score = pd.DataFrame({
    "学号": ["S1", "S2", "S3", "S5"],
    "成绩": [88, 92, 75, 80]
})

# --- 演练 inner merge ---
# inner:只保留两边都有的学号,结果 3 行
inner = pd.merge(df_stu, df_score, on="学号", how="inner")
print("inner 结果, shape:", inner.shape)
print(inner)

# --- 执行过程 ---
# ① pd.merge(df_stu, df_score, on="学号"):指定按"学号"列对齐
# ② how="inner":只取两张表都有的学号,S4 和 S5 被丢弃
# ③ 合并后每行包含左表的"姓名"和右表的"成绩"

**逐行解剖**

- `on="学号"`:指定连接键,两张表都有的列名。如果键名不同,用 `left_on` / `right_on`。
- `how="inner"`:内连接,保留键的交集。最常用,但可能丢数据。
- `shape`:每次 merge 后立即检查 shape,确认行数符合预期。

In [ ]:
# 导入 pandas
import pandas as pd

# 左右表同上
df_stu = pd.DataFrame({
    "学号": ["S1", "S2", "S3", "S4"],
    "姓名": ["张三", "李四", "王五", "赵六"]
})
df_score = pd.DataFrame({
    "学号": ["S1", "S2", "S3", "S5"],
    "成绩": [88, 92, 75, 80]
})

# left:以左表为主,右表匹配不到填 NaN
left = pd.merge(df_stu, df_score, on="学号", how="left")
print("left 结果:")
print(left)
print("---")

# right:以右表为主,左表匹配不到填 NaN
right = pd.merge(df_stu, df_score, on="学号", how="right")
print("right 结果:")
print(right)
print("---")

# outer:取两边学号的并集,匹配不到填 NaN
outer = pd.merge(df_stu, df_score, on="学号", how="outer")
print("outer 结果:")
print(outer)

# --- 执行过程 ---
# ① left:保留 S1-S4 全部行,S4 找不到成绩 -> NaN
# ② right:保留 S1-S3、S5,S5 找不到姓名 -> NaN
# ③ outer:保留 S1-S5 全部行,各自缺失的一侧填 NaN

**逐行解剖**

- **left join**:主表全保留,副表匹配不上就 NaN。适合"基于主表扩展字段"。
- **right join**:把副表当主表,逻辑和 left 反一下。
- **outer join**:取并集,一行不丢但可能大量 NaN。
- 规律:inner 可能丢行,outer 可能增行,一对多匹配会膨胀——务必 merge 后检查 shape。

**练习**

员工表有"员工号/姓名",部门表有"员工号/部门"。请用 left join 把两个表合并,再查看合并后的行数和 NaN 情况,观察哪些员工没有部门信息。

> 问自己:
> - left join 应该把哪个表放左边(主表)?
> - 合并后如何快速找到"没匹配上"的行(提示:NaN 在哪一列)?

In [ ]:
# 学员代码区
import pandas as pd

df_emp = pd.DataFrame({
    "员工号": ["E1", "E2", "E3", "E4"],
    "姓名": ["张三", "李四", "王五", "赵六"]
})
df_dept = pd.DataFrame({
    "员工号": ["E1", "E3", "E5"],
    "部门": ["技术", "市场", "财务"]
})

# 请用 left join 合并,并检查结果
pass

In [ ]:
# 参考答案
import pandas as pd

df_emp = pd.DataFrame({
    "员工号": ["E1", "E2", "E3", "E4"],
    "姓名": ["张三", "李四", "王五", "赵六"]
})
df_dept = pd.DataFrame({
    "员工号": ["E1", "E3", "E5"],
    "部门": ["技术", "市场", "财务"]
})

# 以员工表为主表做 left join
merged = pd.merge(df_emp, df_dept, on="员工号", how="left")
print("合并后行数:", merged.shape[0])
print(merged)

# 找没有部门的员工,部门列为 NaN
print("没匹配上的员工:")
print(merged[merged["部门"].isna()])

#### concat(axis=0/1, ignore_index)

痛点:两个结构相同的表怎么纵向堆叠?两个表列不同,怎么横向拼起来?

类比:concat 就像"粘纸":axis=0 把两张纸上下粘(加行),axis=1 把两张纸左右粘(加列)。

解释: **concat** 沿指定轴拼接多个 DataFrame,**ignore_index=True** 让新生成的索引重新编号。

In [ ]:
# 导入 pandas
import pandas as pd

# 两个同构 DataFrame
df1 = pd.DataFrame({"A": [1, 2], "B": [3, 4]})
df2 = pd.DataFrame({"A": [5, 6], "B": [7, 8]})

# --- 演练:纵向堆叠(axis=0) ---
# ignore_index=True 让新索引 0-3 连续,避免索引重复
stacked = pd.concat([df1, df2], axis=0, ignore_index=True)
print("纵向堆叠结果:")
print(stacked)

# --- 执行过程 ---
# ① [df1, df2]:要拼接的 DataFrame 列表,顺序决定堆叠顺序
# ② axis=0:沿行方向(纵向)堆叠,相当于 df1 下面接 df2
# ③ ignore_index=True:丢弃原索引 0/1,重新生成 0/1/2/3

**逐行解剖**

- `axis=0`:**纵向**,上下堆叠,列对齐、行数增加。这是最常用的"合并两个月份的报表"场景。
- `axis=1`:**横向**,左右拼接,行索引对齐、列数增加。
- `ignore_index=True`:重置索引为连续整数;不设置时可能产生重复索引,影响后续选取。

In [ ]:
# 导入 pandas
import pandas as pd

# 左右拼接示例:列不同
df_left = pd.DataFrame({"A": [1, 2], "B": [3, 4]})
df_right = pd.DataFrame({"C": [5, 6], "D": [7, 8]})

# 横向拼接(axis=1),行索引自动对齐
wide = pd.concat([df_left, df_right], axis=1)
print("横向拼接结果:")
print(wide)

# --- 执行过程 ---
# ① axis=1:沿列方向(横向)拼接
# ② 两张表都是 2 行,索引对齐为 0、1
# ③ 最终得到 A/B/C/D 四列

**逐行解剖**

- 横向拼接时 Pandas 按索引对齐,如果两表行数不同,多出来的格子填 NaN。
- concat 与 merge 区别:concat 只拼形状(不关心键),merge 按 key 对齐(像 JOIN)。想"粘数据"用 concat,想"查关系"用 merge。

**练习**

给定 3 个班级的小表格(列都是"姓名/成绩"),请用 concat 把它们纵向合并成一张年级大表,索引重新编号。

> 问自己:
> - ignore_index 不设为 True 会发生什么?
> - 如果某个班级多了"性别"列,concat 还能直接纵向堆叠吗(列不一致)?

In [ ]:
# 学员代码区
import pandas as pd

df_a = pd.DataFrame({"姓名": ["张三", "李四"], "成绩": [88, 92]})
df_b = pd.DataFrame({"姓名": ["王五"], "成绩": [75]})
df_c = pd.DataFrame({"姓名": ["赵六", "孙七"], "成绩": [85, 90]})

# 请合并三个班级的表,gisc_index 重编号
pass

In [ ]:
# 参考答案
import pandas as pd

df_a = pd.DataFrame({"姓名": ["张三", "李四"], "成绩": [88, 92]})
df_b = pd.DataFrame({"姓名": ["王五"], "成绩": [75]})
df_c = pd.DataFrame({"姓名": ["赵六", "孙七"], "成绩": [85, 90]})

# 纵向堆叠,ignore_index 重编号
all_df = pd.concat([df_a, df_b, df_c], axis=0, ignore_index=True)
print("年级大表,shape:", all_df.shape)
print(all_df)

#### pivot_table

痛点:销售表每行是一条记录,想看"城市 × 季度"的交叉汇总表,怎么写?

类比:长条数据像一根挂面(只有"日期/城市/销量"三列),透视表把它盘成一张二维网格(行=日期、列=城市、值=销量)。

解释: **pivot_table** 把长表重塑为宽表,指定 `index`(行)、`columns`(列)、`values`(值)、`aggfunc`(聚合函数)。

In [ ]:
# 导入 pandas
import pandas as pd

# 长表:每行是一条销售记录
df = pd.DataFrame({
    "日期": ["周一", "周一", "周二", "周二"],
    "城市": ["北京", "上海", "北京", "上海"],
    "销量": [120, 85, 96, 110]
})

# --- 演练 pivot_table ---
pt = df.pivot_table(
    index="日期",
    columns="城市",
    values="销量",
    aggfunc="sum"
)
print("透视表结果:")
print(pt)

# --- 执行过程 ---
# ① index="日期":把"日期"的不同值作为行索引(周一、周二)
# ② columns="城市":把"城市"的不同值作为列(北京、上海)
# ③ values="销量":交叉点的数值来源
# ④ aggfunc="sum":当同一(日期,城市)有多行时,汇总方式

**逐行解剖**

- `index` + `columns`:共同决定一个网格单元。如果某个组合在原始数据中出现多次,就用 `aggfunc` 聚合。
- 缺组合:如果(周一,北京)没有记录,默认填 NaN。可通过 `fill_value=0` 改成 0。
- 与 groupby 区别:groupby 输出"长"表(每行一组),pivot_table 输出"宽"表(矩阵形式)。同一个问题两种写法,选用图表更顺手的。

In [ ]:
# 导入 pandas
import pandas as pd

# 带重复组合的长表
df = pd.DataFrame({
    "日期": ["周一", "周一", "周一", "周二", "周二"],
    "城市": ["北京", "北京", "上海", "北京", "上海"],
    "销量": [120, 80, 85, 96, 110]
})

# 多行同组合时用 aggfunc 汇总
pt_sum = df.pivot_table(
    index="日期", columns="城市", values="销量",
    aggfunc="sum", fill_value=0
)
print("sum 透视表:")
print(pt_sum)
print("---")

# 换一个聚合函数:求均值
pt_mean = df.pivot_table(
    index="日期", columns="城市", values="销量",
    aggfunc="mean", fill_value=0
)
print("mean 透视表:")
print(pt_mean)

# --- 执行过程 ---
# ① 周一对应北京有两行(120 和 80),sum=200、mean=100
# ② fill_value=0:没有数据的组合不再显示 NaN 而是 0

**逐行解剖**

- `aggfunc`:可以是 "sum"、"mean"、"count"、"max" 等字符串,也可以是函数列表 ["sum", "mean"],后者会生成多层列索引。
- `fill_value=0`:透视表中空组合的替代值,对数值型指标常用。
- `margins=True`:给透视表加上行/列汇总,快速看总计。

**练习**

订单表已有"月份、产品、金额"三列,请用 pivot_table 生成"月份 × 产品"的总金额透视表,并让空组合显示为 0。

> 问自己:
> - index、columns、values 各放哪一列?
> - aggfunc 应该选 sum 还是 mean?

In [ ]:
# 学员代码区
import pandas as pd

df = pd.DataFrame({
    "月份": ["1月", "1月", "2月", "2月", "2月"],
    "产品": ["A", "B", "A", "B", "B"],
    "金额": [100, 200, 150, 80, 120]
})

# 请生成月份 x 产品 的透视表
pass

In [ ]:
# 参考答案
import pandas as pd

df = pd.DataFrame({
    "月份": ["1月", "1月", "2月", "2月", "2月"],
    "产品": ["A", "B", "A", "B", "B"],
    "金额": [100, 200, 150, 80, 120]
})

pt = df.pivot_table(
    index="月份", columns="产品", values="金额",
    aggfunc="sum", fill_value=0
)
print(pt)

#### fillna / dropna / isnull

痛点:合并表经常遇到 NaN,这些空值怎么处理?删还是填?

类比:学生档案里"家庭电话"一栏很多人没填,你可以选择删掉这些学生(不现实)、补"未知"、或用班级平均年龄偷填一下。

解释: **isnull** 检测缺失值,**dropna** 删除含缺失值的行/列,**fillna** 填充缺失值。处理流程:发现 → 删除 or 填充。

In [ ]:
# 导入 pandas 和 numpy(NaN 字面量来自 numpy)
import pandas as pd
import numpy as np

# 创建含缺失值的 DataFrame
df = pd.DataFrame({
    "姓名": ["张三", "李四", "王五", "赵六"],
    "年龄": [18, np.nan, 19, np.nan],
    "成绩": [88, 92, np.nan, 85]
})

# --- 演练:发现缺失值 ---
# isnull() 返回同形布尔矩阵,True 表示为空
print("isnull 布尔矩阵:")
print(df.isnull())
print("---")

# isnull().sum():每列缺失多少个
print("每列缺失数:")
print(df.isnull().sum())

# --- 执行过程 ---
# ① np.nan:Python 的"不是一个数",在 Pandas 里会被识别为缺失
# ② df.isnull():逐元素判断,True 表示该位置缺失
# ③ .sum():True 当 1、False 当 0,sum 得到每列缺失计数

**逐行解剖**

- 缺失值来源:merge 匹配不到、爬取失败、用户未填、文件中是空字符串(注意:空字符串 "" 不算 np.nan)。
- 数据清洗第一步永远是 `df.isnull().sum()`,先看清缺失规模再决定怎么处理。

In [ ]:
# 导入 pandas 和 numpy
import pandas as pd
import numpy as np

# 创建含缺失值的 DataFrame
df = pd.DataFrame({
    "姓名": ["张三", "李四", "王五", "赵六"],
    "年龄": [18, np.nan, 19, np.nan],
    "成绩": [88, 92, np.nan, 85]
})

# --- 演练 dropna:删除含缺失值的行 ---
# dropna() 默认 axis=0,任何一行有 NaN 就整行删除
dropped = df.dropna()
print("dropna 后:")
print(dropped)

# --- 执行过程 ---
# ① 李四(年龄 NaN)被删
# ② 王五(成绩 NaN)被删
# ③ 剩余张三、赵六,赵六仍有年龄 NaN?不对,赵六年龄也是 NaN -> 只剩张三
# 最终只剩没有任何 NaN 的行

**逐行解剖**

- `dropna()`:默认 `how="any"`,只要有一个 NaN 就删整行。`how="only"`要求全部都是 NaN 才删。
- `subset=["年龄"]`:只在指定列里判断 NaN,其他列不管。
- 删除是"暴力"处理,数据少时慎用。

In [ ]:
# 导入 pandas 和 numpy
import pandas as pd
import numpy as np

# 创建含缺失值的 DataFrame
df = pd.DataFrame({
    "姓名": ["张三", "李四", "王五", "赵六"],
    "年龄": [18, np.nan, 19, np.nan],
    "成绩": [88, 92, np.nan, 85]
})

# --- 演练 fillna:填充缺失值 ---
# 用固定值填充
print("填 0:")
print(df.fillna(0))
print("---")

# 用该列均值填充(常用)
print("填列均值:")
print(df.fillna(df.mean(numeric_only=True)))
print("---")

# 前向填充(用前一个有效值)
print("ffill 前向填充:")
print(df.ffill())

# --- 执行过程 ---
# ① df.mean(numeric_only=True):只对数值列求均值,(18+19)/2=18.5、(88+92+85)/3≈88.3
# ② fillna(dict/Series/标量):按对应关系把 NaN 替换
# ③ ffill:前向填充,每 NaN 用它"头顶上"最近的非 NaN 替代

**逐行解剖**
- `fillna(0)`:最粗暴,适合"没有就按 0 算"的业务。
- `fillna(df.mean())`:按列均值填充,是建模前的常用手法,保持总体均值不变。
- `ffill()` 即 forward-fill,适合时间序列(用上一个时刻的值)。
- 注意:fillna、dropna 默认**不修改原数据**(inplace 默认为 False),需要赋值或加 `inplace=True`。

**练习**

"员工工龄"字段中有 3 个 NaN,请先用 isnull().sum() 查看缺失数量,再用"列均值"填充缺失值,最后验证是否还有 NaN。

> 问自己:
> - df.mean() 默认包括姓名列怎么办?(numeric_only)
> - fillna 后会修改原 DataFrame 吗?如何保存结果?

In [ ]:
# 学员代码区
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "姓名": ["张三", "李四", "王五", "赵六", "孙七"],
    "工龄": [3, np.nan, 7, np.nan, np.nan]
})

# 任务 1:查看每列缺失数
# 任务 2:用均值填充
# 任务 3:验证是否还有 NaN
pass

In [ ]:
# 参考答案
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "姓名": ["张三", "李四", "王五", "赵六", "孙七"],
    "工龄": [3, np.nan, 7, np.nan, np.nan]
})

# 任务 1
print("每列缺失数:")
print(df.isnull().sum())

# 任务 2:用均值填充,numeric_only 避免对姓名求均值
df_filled = df.fillna(df.mean(numeric_only=True))
print("填充后:")
print(df_filled)

# 任务 3
print("还有 NaN 吗?", df_filled.isnull().sum().sum() > 0)

**今日小结**

| 概念 | 解决的痛点 |
|---|---|
| groupby + agg | 按类别一次性输出多个统计量,不用写 N 行 query |
| merge(inner/left/right/outer) | 像 SQL JOIN 一样把两张表按 key 拼起来 |
| concat(axis=0/1) | 快速纵向堆叠或横向拼多个同形/异形表 |
| pivot_table | 把长表重塑成"行 × 列"的二维交叉表 |
| fillna / dropna / isnull | 检测缺失值、决定删还是填 |

**更多练习**
- 当堂练:course/lesson14/in_class/practice01.py ~ practice06.py
- 课后作业:course/lesson14/homework/task01.py ~ task03.py